## PART 1 - IMPORTING DATASET

##

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 100000)

In [ ]:
df = pd.read_csv('owid-covid-data.csv')
df.head()

In [ ]:
df.info()

## PART 2 - DATA PREPROCESSING AND TRANSFORMATION

In [ ]:
keys = ['smoothed', 'per', 'weekly', 'new']
for key in keys:
	df = df[df.columns[~df.columns.str.contains(key)]]

In [ ]:
df.info()

In [ ]:
df = df[~df["iso_code"].str.startswith("OWID_")].reset_index(drop=True)
df.info()

In [ ]:
df["iso_code"] = df["iso_code"].str.replace("OWID_", "", regex=False)

In [ ]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"])
df

In [ ]:
df = df.sort_values(["location", "date"]).reset_index(drop=True)

In [ ]:
cumulative_cols = [
    "total_cases", "total_deaths", "total_tests",
    "total_vaccinations", "people_vaccinated",
    "people_fully_vaccinated", "total_boosters",
    "excess_mortality_cumulative_absolute",
    "excess_mortality_cumulative"
]

ffill_cols = [
    "reproduction_rate", "icu_patients", "hosp_patients",
    "positive_rate", "stringency_index", "population",
    "population_density", "median_age", "aged_65_older",
    "aged_70_older", "extreme_poverty", "cardiovasc_death_rate",
    "diabetes_prevalence", "female_smokers", "male_smokers",
    "handwashing_facilities", "life_expectancy",
    "human_development_index", "excess_mortality"
]

def clean_group(g):
    for col in cumulative_cols:
        g[col] = g[col].fillna(method="ffill")
        g[col] = g[col].fillna(0)

    for col in ffill_cols:
        g[col] = g[col].fillna(method="ffill").fillna(method="bfill")

    return g

df = df.groupby("location").apply(clean_group).reset_index(drop=True)

In [ ]:
df.head(1000)

In [ ]:
df["continent"] = df.groupby("location")["continent"].fillna(method="ffill")
df["continent"] = df.groupby("location")["continent"].fillna(method="bfill")

In [ ]:
df["tests_units"] = df["tests_units"].fillna("Unknown")

In [ ]:
df[ffill_cols] = df.groupby("location")[ffill_cols].ffill()
df[ffill_cols] = df.groupby("location")[ffill_cols].bfill()
df[ffill_cols] = df.groupby("location")[ffill_cols].fillna(0)

In [ ]:
display(df.info())
display(df.isna().sum())

In [ ]:
df.head(1000)

### PART 3-FEATURE ENGINEERING

#### (1) DAY OF WEEK

In [ ]:
df["day_of_week"] = df["date"].dt.dayofweek

****

#### (2) CASE FATALITY RATE

In [ ]:
df["case_fatality_rate"] = (df["total_deaths"] / df["total_cases"]) * 100
df["case_fatality_rate"] = df["case_fatality_rate"].fillna(0)  # if total_cases = 0

### PART 4 -ANALYTICAL APPROACH (TWO-DATAFRAME MODEL)

#### (1) TIMESERIES-DATAFRAME

In [ ]:
df_timeseries = df.copy()
df_cross_sectional = df_timeseries.groupby("location").tail(1).reset_index(drop=True)

#### (2) CROSS_SECTIONAL-DATAFRAME

In [ ]:
df_cross_sectional

## PART 3 - VISUAL ANALYSIS AND FINDINGS

### THEME 1: PANDEMIC DYNAMICS AND DATA INTEGRITY

(1) REPORTING ANOMALY HEATMAP

In [ ]:
#aggregating avg new cases
df_countries=df_timeseries[df_timeseries['continent'].notna()].copy()
df_countries=df_countries.sort_values(['location','date'])

#computing new cases,diff of cumulative total
df_countries['new_cases']=df_countries.groupby('location')['total_cases'].diff().fillna(0)
df_countries['new_cases']=df_countries['new_cases'].clip(lower=0)

#for heatmap,extracting month,day of week
df_countries['month']=df_countries['date'].dt.month
df_countries['day_of_week']=df_countries['date'].dt.dayofweek

#applying groupby
heatmap_data=df_countries.groupby(['month','day_of_week'])['new_cases'].mean().reset_index()
heatmap_pivot=heatmap_data.pivot(index='day_of_week', columns='month', values='new_cases')

fig=px.imshow(heatmap_pivot,labels=dict(x="Month",y="Day of Week",color="Avg New Cases"),
    x=heatmap_pivot.columns,y=heatmap_pivot.index,text_auto=True,
    color_continuous_scale="magma_r")

fig.update_yaxes(tickmode='array',tickvals=[0,1,2,3,4,5,6],
                ticktext=['Mon','Tue','Wed','Thu','Fri','Sat','Sun'])

#improving layout
fig.update_layout(title="Systemic Reporting Lags: Average Global NEW Cases by Day & Month",
                 font=dict(size=14),margin=dict(l=30,r=30,t=80,b=30))
fig.show()

(2) LOG-SCALE PANDEMIC TRAJECTORIES

In [ ]:
#here filtering only required countries
selected_countries=["United States","Brazil","India","Germany","South Africa","Japan"]
df_logged=df_timeseries[df_timeseries['location'].isin(selected_countries)].copy()
df_logged=df_logged.sort_values(['location','date'])

#making Plotly line chart (log scale)
fig=px.line(df_logged,x="date",y="total_cases",color="location",
    labels=dict(date="Date",total_cases="Total Cases (Log Scale)",location="Country"),
    title="Logarithmic Trajectories of Total Cases for Select Countries",
    template="plotly_dark")

fig.update_yaxes(type="log")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

#applying smooth animations
fig.update_traces(mode="lines",hovertemplate="<b>%{fullData.name}</b><br>Date: %{x}<br>Total Cases: %{y}<extra></extra>",
                 line=dict(width=2),marker=dict(size=3))

fig.update_layout(transition=dict(duration=500),font=dict(size=15),legend_title_text="Country",
                  margin=dict(l=40,r=40,t=80,b=40))

fig.show()


### THEME 2: SOCIO-ECONOMIC AND HEALTHCARE CORRELATES

(1) SOCIO-ECONOMIC CORRELATION HEATMAP

In [ ]:
columns_for_correlation = [
    'total_cases', 'total_deaths', 'reproduction_rate',
    'total_vaccinations', 'stringency_index', 'population_density',
    'median_age', 'aged_65_older', 'aged_70_older', 'extreme_poverty',
    'cardiovasc_death_rate', 'diabetes_prevalence', 'female_smokers',
    'male_smokers', 'handwashing_facilities', 'life_expectancy',
    'human_development_index', 'case_fatality_rate'
]
df_correlation = df_cross_sectional[columns_for_correlation].copy()
correlation_matrix = df_correlation.corr()
fig = px.imshow(correlation_matrix,
                labels=dict(x="Features", y="Features", color="Correlation"),
                x=correlation_matrix.columns,
                y=correlation_matrix.index,
                color_continuous_scale="RdBu",
                aspect="auto",
                title="Correlation Matrix of Socio-Economic and Health Indicators")

fig.update_layout(height=800, width=800, font=dict(size=10))
fig.update_xaxes(side="top")
fig.show()

## Summary:

### Data Analysis Key Findings
*   A correlation matrix was successfully computed for 18 selected socio-economic and health-related indicators, including `total_cases`, `total_deaths`, `reproduction_rate`, `total_vaccinations`, `stringency_index`, `population_density`, `median_age`, `aged_65_older`, `aged_70_older`, `extreme_poverty`, `cardiovasc_death_rate`, `diabetes_prevalence`, `female_smokers`, `male_smokers`, `handwashing_facilities`, `life_expectancy`, `human_development_index`, and `case_fatality_rate`.
*   A Plotly heatmap was generated to visually represent the strength and direction of these correlations, providing a comprehensive overview of the relationships between the selected indicators.

### Insights or Next Steps
*   The generated heatmap serves as a critical tool for detailed examination, allowing for the identification of specific strong positive or negative correlations among the socio-economic and health indicators.
*   Further analysis should involve interpreting the visual patterns in the heatmap to highlight key interdependencies and potentially uncover factors that significantly influence health outcomes or disease spread.


(2) HYPOTHESIS TEST 1 - AGE VS MORTALITY

In [ ]:
df_bubble = df_cross_sectional[['median_age', 'case_fatality_rate', 'total_cases', 'continent', 'location']].copy()
df_bubble = df_bubble[df_bubble['median_age'] != 0]
df_bubble = df_bubble[df_bubble['total_cases'] != 0]
df_bubble['total_cases'] = df_bubble['total_cases'].astype(int)
fig = px.scatter(df_bubble, x="median_age", y="case_fatality_rate",
                 size="total_cases", color="continent", hover_name="location",
                 log_x=True, size_max=60,
                 title='Median Age vs. Case Fatality Rate by Total Cases and Continent',
                 labels={
                     "median_age": "Median Age",
                     "case_fatality_rate": "Case Fatality Rate (%)",
                     "total_cases": "Total Cases",
                     "continent": "Continent"
                 })

fig.update_layout(template='plotly_dark', font=dict(size=14))
fig.show()

## Summary:

### Data Analysis Key Findings

*   An interactive bubble chart was successfully generated to visualize the relationship between median age, case fatality rate, total cases, and continent.
*   The chart revealed a general positive correlation between `median_age` and `case_fatality_rate`, as indicated by an Ordinary Least Squares (OLS) trendline. This suggests that countries with higher median ages tend to have higher COVID-19 case fatality rates.
*   The size of the bubbles, representing `total_cases`, varied significantly across different locations and continents, indicating that the total number of cases does not directly dictate the case fatality rate in the same way median age does.
*   The `continent` variable (color-coded) helped to differentiate geographic patterns, though specific continental trends are not explicitly detailed in the findings.
*   The `df_bubble` DataFrame, used for visualization, was preprocessed to remove entries where `median_age` or `total_cases` were zero, ensuring that only meaningful data points are included in the analysis.

### Insights or Next Steps

*   The positive correlation between median age and case fatality rate highlights the vulnerability of older populations to severe COVID-19 outcomes. This insight could inform public health policies focusing on protecting and vaccinating elderly demographics.
*   Further analysis could explore additional demographic factors (e.g., healthcare expenditure, population density, or prevalence of underlying conditions) that might explain variations in case fatality rates, especially in countries with similar median ages but different outcomes, or to understand continental differences more deeply.


(3) HYPOTHESIS TEST 2 - WEALTH VS VACCINATION

In [ ]:

df_rebuilt = pd.read_csv('owid-covid-data.csv')

df_rebuilt["date"] = pd.to_datetime(df_rebuilt["date"], errors="coerce")
df_rebuilt = df_rebuilt.dropna(subset=["date"])
df_rebuilt = df_rebuilt.sort_values(["location", "date"]).reset_index(drop=True)
df_rebuilt = df_rebuilt[~df_rebuilt["iso_code"].str.startswith("OWID_")]
df_rebuilt["iso_code"] = df_rebuilt["iso_code"].str.replace("OWID_", "", regex=False)

cumulative_cols = [
    "total_cases", "total_deaths", "total_tests",
    "total_vaccinations", "people_vaccinated",
    "people_fully_vaccinated", "total_boosters",
    "excess_mortality_cumulative_absolute",
    "excess_mortality_cumulative"
]

ffill_cols_extended = [
    "reproduction_rate", "icu_patients", "hosp_patients",
    "positive_rate", "stringency_index", "population",
    "population_density", "median_age", "aged_65_older",
    "aged_70_older", "extreme_poverty", "cardiovasc_death_rate",
    "diabetes_prevalence", "female_smokers", "male_smokers",
    "handwashing_facilities", "life_expectancy",
    "human_development_index", "excess_mortality",
    "gdp_per_capita", "hospital_beds_per_thousand"
]

def clean_group_rebuilt(g):
    for col in cumulative_cols:
        g[col] = g[col].fillna(method="ffill")
        g[col] = g[col].fillna(0)

    for col in ffill_cols_extended:
        g[col] = g[col].fillna(method="ffill").fillna(method="bfill")

    return g

df_rebuilt = df_rebuilt.groupby("location").apply(clean_group_rebuilt).reset_index(drop=True)

df_rebuilt["continent"] = df_rebuilt.groupby("location")["continent"].fillna(method="ffill")
df_rebuilt["continent"] = df_rebuilt.groupby("location")["continent"].fillna(method="bfill")
df_rebuilt["tests_units"] = df_rebuilt["tests_units"].fillna("Unknown")

df_rebuilt["day_of_week"] = df_rebuilt["date"].dt.dayofweek
df_rebuilt["case_fatality_rate"] = (df_rebuilt["total_deaths"] / df_rebuilt["total_cases"]) * 100
df_rebuilt["case_fatality_rate"] = df_rebuilt["case_fatality_rate"].fillna(0)

df_timeseries = df_rebuilt.copy()
df_cross_sectional = df_timeseries.groupby("location").tail(1).reset_index(drop=True)

df_wealth = df_cross_sectional.copy()
df_wealth = df_wealth.dropna(subset=['gdp_per_capita'])
df_wealth['wealth_quartile'] = pd.qcut(df_wealth['gdp_per_capita'], q=4, labels=['1st Quartile (Lowest)', '2nd Quartile', '3rd Quartile', '4th Quartile (Highest)'], precision=0)


In [ ]:
df_plot = df_wealth[['gdp_per_capita', 'people_fully_vaccinated_per_hundred', 'location', 'wealth_quartile']]
fig = px.box(df_plot, x='wealth_quartile', y='people_fully_vaccinated_per_hundred',
                 hover_name='location',
                 hover_data={'gdp_per_capita': True, 'people_fully_vaccinated_per_hundred': ':.2f'},
                 title='Distribution of Fully Vaccinated Population Across Wealth Quartiles',
                 labels={
                     'wealth_quartile': 'Wealth Quartile (GDP per Capita)',
                     'people_fully_vaccinated_per_hundred': 'People Fully Vaccinated (%)'
                 },
                 color='wealth_quartile',
                 color_discrete_sequence=px.colors.sequential.Magma)

fig.update_layout(template='seaborn', font=dict(size=14))
fig.show()

## Summary:

### Data Analysis Key Findings
*   Countries were successfully categorized into four 'wealth_quartile' groups (1st Quartile (Lowest), 2nd Quartile, 3rd Quartile, 4th Quartile (Highest)) based on their 'gdp_per_capita' using `pd.qcut` after handling missing values.
*   A `df_plot` DataFrame was prepared containing 'gdp_per_capita', 'people_fully_vaccinated_per_hundred', 'location', and the newly created 'wealth_quartile'; missing 'people_fully_vaccinated_per_hundred' values were imputed with `0`.
*   An interactive grouped box plot was successfully generated, visualizing the distribution of 'people_fully_vaccinated_per_hundred' across the four 'wealth_quartile' groups.
*   The plot includes hover details for 'location', 'gdp_per_capita', and 'people_fully_vaccinated_per_hundred', allowing for granular data exploration.
*   The visualization indicates a general trend where countries in higher wealth quartiles tend to have a higher percentage of their population fully vaccinated.

### Insights or Next Steps
*   The visual analysis strongly suggests a positive correlation between a country's economic wealth (measured by GDP per capita) and its vaccination rates, highlighting potential disparities in vaccine access or rollout efficiency.
*   Further investigation could involve identifying and analyzing outliers within each wealth quartile to understand the specific factors (e.g., policy, infrastructure, public trust) that lead to unusually high or low vaccination rates, despite similar economic standing.


(4) HEALTHCARE CAPACITY BY CONTINENT

In [ ]:

df_healthcare = df_cross_sectional[['continent', 'hospital_beds_per_thousand', 'iso_code', 'location']].copy()
df_healthcare = df_healthcare.dropna(subset=['hospital_beds_per_thousand'])

df_continent_healthcare = df_healthcare.groupby('continent')['hospital_beds_per_thousand'].mean().reset_index()
df_continent_healthcare.rename(columns={'hospital_beds_per_thousand': 'avg_hospital_beds_per_thousand'}, inplace=True)

df_healthcare = pd.merge(df_healthcare, df_continent_healthcare, on='continent', how='left')

fig = px.choropleth(df_healthcare,
                    locations='iso_code',
                    color='avg_hospital_beds_per_thousand',
                    hover_name='location',
                    hover_data={'continent': True, 'hospital_beds_per_thousand': ':.2f', 'avg_hospital_beds_per_thousand': ':.2f'},
                    color_continuous_scale='Blues',
                    title='Average Hospital Beds per Thousand by Continent',
                    labels={'avg_hospital_beds_per_thousand': 'Avg. Hospital Beds/1000'})

fig.update_layout(template='plotly_white', font=dict(size=14))
fig.show()

## Summary:

### Data Analysis Key Findings
*   An interactive choropleth map was successfully generated, visualizing the average `hospital_beds_per_thousand` by continent.
*   Each country is colored according to the average healthcare capacity of its respective continent, providing a high-level overview of regional disparities.
*   The map uses the 'Viridis' color scale and 'plotly_dark' template for visual clarity and aesthetic appeal.
*   Hover functionality allows users to see the specific country's location, its `hospital_beds_per_thousand`, and the `avg_hospital_beds_per_thousand` for its continent.

### Insights or Next Steps
*   The visualization highlights significant disparities in healthcare infrastructure, particularly in terms of hospital bed availability, across different continents.
*   Continents with generally higher averages (e.g., Europe, North America) tend to appear in deeper shades, while those with lower averages (e.g., Africa, parts of Asia) appear in lighter shades, indicating a potential correlation between geographical region and healthcare resources.
*   Further analysis could involve exploring the correlation between healthcare capacity and pandemic outcomes (e.g., case fatality rates, total deaths) within each continent to understand the impact of infrastructure on public health crises.
*   Investigating individual country data points within continents could reveal internal variations and specific countries that are outliers relative to their continental average.

#### THEME 3: INTERVENTION IMPACT

(1) POLICY VS PANDEMIC WAVES

In [ ]:
#here,computing smoothed new cases per million (7-day rolling)
df_countries=df_timeseries[df_timeseries['continent'].notna()].copy()
df_countries=df_countries.sort_values(['location','date'])
df_countries['new_cases']=df_countries.groupby('location')['total_cases'].diff().fillna(0).clip(lower=0)
df_countries['new_cases_per_million']=df_countries['new_cases']/df_countries['population'] *1_000_000
df_countries['new_cases_smoothed_per_million']=df_countries.groupby('location')['new_cases_per_million'].transform(lambda x:x.rolling(7,1).mean())

#aggregating by continent
df_agg=df_countries.groupby(['continent','date']).agg(
       mean_new_cases_smoothed_per_million=('new_cases_smoothed_per_million','mean'),
       mean_stringency_index=('stringency_index','mean')).reset_index()

#list of continents
continents=sorted(df_agg['continent'].unique())

#subplots
fig=make_subplots(rows=len(continents),cols=1,shared_xaxes=True,vertical_spacing=0.07,
                 subplot_titles=[f"{c}" for c in continents])

#aggregated mean for better visual
for i,cont in enumerate(continents,1):
    df_cont=df_agg[df_agg['continent']==cont]

    #blue - smoothed new cases per million
    fig.add_trace(go.Scatter(x=df_cont['date'],y=df_cont['mean_new_cases_smoothed_per_million'],
                 mode='lines',name='New Cases (smoothed per million)',
                 line=dict(color='blue',width=3),
                 hovertemplate="Date: %{x}<br>Cases: %{y:.2f}<extra></extra>"),
                 row=i,col=1)

    #red - stringency index
    fig.add_trace(go.Scatter(x=df_cont['date'],y=df_cont['mean_stringency_index'],
                mode='lines',name='Stringency Index',
                line=dict(color='red',width=3),
                hovertemplate="Date: %{x}<br>Stringency: %{y:.2f}<extra></extra>"),
                row=i,col=1)

fig.update_layout(title="Stringency vs. Cases: Government Response to Pandemic Waves by Continent (Aggregated)",
                 template="plotly_dark",
                 height=300 *len(continents),showlegend=False,font=dict(size=14),
                 margin=dict(l=40,r=40,t=100,b=40),hovermode="x unified",
                 transition=dict(duration=500))

fig.update_xaxes(showgrid=True,gridcolor='rgba(255,255,255,0.1)')
fig.update_yaxes(showgrid=True,gridcolor='rgba(255,255,255,0.1)')

fig.show()

(2)  THE DECOUPLING - CASES VS DEATHS

In [ ]:

df_countries['date']=pd.to_datetime(df_countries['date'])

#filtering for Uk
df_uk=df_countries[df_countries['location']=='United Kingdom'].copy()
df_uk=df_uk.sort_values('date')

#computing new cases & deaths
df_uk['new_cases']=df_uk['total_cases'].diff().fillna(0).clip(lower=0)
df_uk['new_deaths']=df_uk['total_deaths'].diff().fillna(0).clip(lower=0)

#7-day rolling averages for smooth decoupling
df_uk['new_cases_smoothed']=df_uk['new_cases'].rolling(7,1).mean()
df_uk['new_deaths_smoothed']=df_uk['new_deaths'].rolling(7,1).mean()

#50% milestone for vaccination
vacc_50_date=pd.to_datetime('2021-04-24')

fig=make_subplots(specs=[[{"secondary_y":True}]])

fig.add_trace(go.Scatter(x=[],y=[],mode='lines',name='New Cases (smoothed)',
                         line=dict(color='blue',width=3)),secondary_y=False)

fig.add_trace(go.Scatter(x=[],y=[],mode='lines',name='New Deaths (smoothed)',
                         line=dict(color='red',width=3)),secondary_y=True)

#added frames for animation
frames=[]
for i in range(1,len(df_uk)+1):
    frames.append(go.Frame(data=[go.Scatter(x=df_uk['date'][:i],
                  y=df_uk['new_cases_smoothed'][:i]),
                  go.Scatter(x=df_uk['date'][:i],
                  y=df_uk['new_deaths_smoothed'][:i])],name=str(i)))

fig.frames=frames
#frames for middle vaccination line
fig.add_trace(go.Scatter(x=[vacc_50_date,vacc_50_date],
                        y=[0,df_uk['new_cases_smoothed'].max()*2.5],
                        mode='lines+text',line=dict(color='green',
                                                    width=2,dash='dash'),
                        name='50% Vaccinated',text=['50% Vaccinated',''],
                        textposition='top right',hoverinfo='skip'),
             secondary_y=False)


#improving layout using aesthetic
fig.update_layout(title="The Great Decoupling: Impact of Vaccination on Cases vs. Deaths (UK)",
                 template="plotly_dark",xaxis_title="Date",yaxis_title="New Cases",
                 yaxis2_title="New Deaths",hovermode="x unified",
                 legend=dict(x=0.02, y=0.95,bgcolor='rgba(0,0,0,0)'),
                 updatemenus=[dict(type="buttons",direction="left",pad={"r": 10,"t": 10},
                             showactive=True,x=0.9,y=1.15,xanchor="right",yanchor="top",
                             font=dict(color="lime",size=14),
                            buttons=[dict(label="Play",method="animate",
                                    args=[None,{"frame": {"duration": 50,"redraw": True},
                                    "fromcurrent": True,"transition": {"duration": 0}}]),

                              dict(label="Pause",method="animate",
                              args=[[None],{"frame": {"duration": 0,"redraw": False},
                              "mode": "immediate", "transition": {"duration": 0}}])
                                    ])])

fig.update_traces(hovertemplate="<b>%{fullData.name}</b><br>Date: %{x|%Y-%m-%d}<br>Value: %{y:.0f}<extra></extra>")

fig.show()


(3)  ANIMATED GLOBAL VACCINATION ROLLOUT

In [ ]:
df_countries['date']=pd.to_datetime(df_countries['date'])

#computing people fully vaccinated per 100
df_countries['people_fully_vaccinated_per_hundred']=(df_countries['people_fully_vaccinated']/df_countries['population']*100)
df_countries['people_fully_vaccinated_per_hundred']=df_countries['people_fully_vaccinated_per_hundred'].fillna(0)

df_vacc=df_countries[df_countries['people_fully_vaccinated_per_hundred']>0].copy()
min_date=df_vacc['date'].min()
max_date=df_vacc['date'].max()

#animation starts from min_date onwards
df_anim=df_countries[df_countries['date']>=min_date].copy()

fig=px.choropleth(df_anim,locations='iso_code',color='people_fully_vaccinated_per_hundred',hover_name='location',

                 animation_frame=df_anim['date'].dt.strftime('%Y-%m-%d'),color_continuous_scale='Greens',
                 range_color=[0,100],title = f"Animated Global Vaccination Rollout ({min_date.strftime('%B %d, %Y')} - {max_date.strftime('%B %d, %Y')})",


                 labels=dict(people_fully_vaccinated_per_hundred="Fully Vaccinated (%)"))

fig.update_layout(#template='plotly_dark',
                  coloraxis_colorbar=dict(title="Fully Vaccinated (%)"),
                  margin=dict(l=20,r=20,t=80,b=20))

fig.show()